# 🦺 Argus Safety AI — Fresh Training from Scratch
**Configuration**: Fresh Start (Epoch 1 to 25) | `imgsz=800` | YOLOv8 Medium (`yolov8m.pt`)
**Dataset**: 22,138 Images | 52,008 Annotations | 6 Canonical PPE Classes
**Estimated Duration**: ~2 Hours on Colab T4 GPU
**Target Accuracy**: **~88% – 91% mAP@50**, >90% Precision on Core PPE

Follow the numbered steps below from top to bottom.

In [ ]:
# STEP 1: Verify GPU & Install Ultralytics
!nvidia-smi
!pip install -q ultralytics

In [ ]:
# STEP 2: Connect Your Google Drive (for checkpoint backups)
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')
drive_workspace = Path('/content/drive/MyDrive/argus_training')
drive_workspace.mkdir(parents=True, exist_ok=True)
print('Google Drive connected successfully!')

In [ ]:
# STEP 3: Unzip Dataset to Local High-Speed Storage
# Ensure argus_ppe_dataset.zip is in your Google Drive root
!mkdir -p /content/dataset

zip_path = '/content/drive/MyDrive/argus_ppe_dataset.zip'
if os.path.exists(zip_path):
    print('Extracting dataset (this takes about 1-2 minutes)...')
    !unzip -q "$zip_path" -d /content/dataset/
    print('✅ Dataset extracted successfully!')
else:
    print(f'❌ ERROR: {zip_path} not found!')
    print('Please upload argus_ppe_dataset.zip to the main folder of your Google Drive.')

In [ ]:
# STEP 4: Configure Canonical data.yaml
data_yaml_content = """
path: /content/dataset
train: train/images
val:   valid/images
test:  test/images

nc: 6
names: ['Gloves', 'Vest', 'goggles', 'helmet', 'mask', 'safety_shoe']
"""

with open('/content/dataset/data.yaml', 'w') as f:
    f.write(data_yaml_content.strip())

print('✅ data.yaml configured successfully!')

In [ ]:
# STEP 5: Start FRESH Training from Epoch 1 to 25 (~2 Hours)
from ultralytics import YOLO

# Clean fresh start from pre-trained YOLOv8 Medium
model = YOLO('yolov8m.pt')

print('Starting fresh training: Epoch 1 of 25...')
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=25,                  # 25 Epochs Sweet Spot (~2 hours)
    batch=16,
    imgsz=800,                  # High-resolution to resolve goggles, gloves, shoes
    device=0,
    patience=8,                 # Early stop if converged early
    save=True,
    save_period=1,              # Sync checkpoint to Google Drive every epoch
    project='/content/drive/MyDrive/argus_training',
    name='ppe_sweet_spot_fresh', # 100% fresh directory name
    exist_ok=True,
    amp=True,                   # Automatic Mixed Precision for 2x speed
    cos_lr=True,                # Cosine learning rate schedule
    close_mosaic=5,             # Disable mosaic in last 5 epochs for crisp boxes
    seed=42,
    workers=4
)

In [ ]:
# STEP 6: Benchmark on Held-Out Test Split (2,261 Images)
best_model_path = '/content/drive/MyDrive/argus_training/ppe_sweet_spot_fresh/weights/best.pt'
eval_model = YOLO(best_model_path)

metrics = eval_model.val(
    data='/content/dataset/data.yaml',
    split='test',
    imgsz=800,
    conf=0.45,                  # 0.45 confidence threshold for 90%+ Precision
    iou=0.60
)

print('=' * 70)
print('  FINAL EVALUATION BENCHMARK')
print('=' * 70)
print(f'  Mean Precision (all classes): {metrics.box.mp * 100:.2f}%')
print(f'  Mean Recall (all classes):    {metrics.box.mr * 100:.2f}%')
print(f'  mAP@50:                       {metrics.box.map50 * 100:.2f}%')
print(f'  mAP@50-95:                    {metrics.box.map * 100:.2f}%')
print('=' * 70)

In [ ]:
# STEP 7: Download Final Model Weights
import shutil
from google.colab import files

output_path = '/content/drive/MyDrive/argus_training/ppe_merged_best.pt'
shutil.copyfile(best_model_path, output_path)
print(f'✅ Model weights saved to Google Drive: {output_path}')

# Trigger browser download to your local computer
files.download(output_path)
print('Once downloaded, move ppe_merged_best.pt into ml/weights/ in your Argus repository.')